## Plan
- Run *ascairn* and *HOROSCOPE* on the same samples and compare the results. 
- Use samples with short and long read data
- This is e.g. the set of 272 1kGP samples
- Aim for samples with assemblies from the HPRC project
----
### ToDo
- I have the data for all 1kGP samples with HOROSCOPE
- Run  *ascairn* of sample X (download cram; run ascairn; read results)
- Transform *ascairn* call into (a) HOROSCOPE matching haplogroup and (b) calculate mean length from matching ascairn calls
- Combine with 1kGP HOROSCOPE calls
- Result: Sample | HOROSCOPE_H1 | HOROSCOPE_H2 | HOROSCOPE_LENGTH | ascairn_H1 | ascairn_H2 | ascairn_LENGTH | assembly_H1 | assembly_H2 | assembly_LENGTH

## Gather samples

Use this table for the links: https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/data_collections/1000G_2504_high_coverage/1000G_2504_high_coverage.sequence.index

In [ ]:
import pandas as pd
import os
from io import StringIO

In [14]:
### matching samples with long and short read data
matching_samples = [x.split(".")[0] for x in os.listdir("/scratch/hain/centromere/1kgp_unitigs")]

In [62]:
### reformat the related index table...
path = "1000G_698_related_high_coverage.sequence.index"

with open(path) as file:
    lines = [line for line in file if not line.startswith("##")]

header = lines[0].rstrip("\n").split("\t")
rows = [line.rstrip("\n").split("\t") for line in lines[1:]]

# Remove the extra empty field after POPULATION where present.
population_index = header.index("POPULATION")
rows = [
    row[:population_index + 1] + row[population_index + 2:]
    if len(row) == len(header) + 1 and row[population_index + 1] == ""
    else row
    for row in rows
]

df_698 = pd.DataFrame(rows, columns=header)

In [56]:
samplesheet_1kgp = pd.concat([
    pd.read_csv("1000G_2504_high_coverage.sequence.index", comment="#", sep="\t", header=0),
    df_698
    ])

In [60]:
print(f"Unitig samples: {len(matching_samples)}")
print(f"1kGP samples: {len(samplesheet_1kgp)}")
print(f"Matching samples: {len([x for x in matching_samples if x in samplesheet_1kgp['SAMPLE_NAME'].values])}")

Unitig samples: 272
1kGP samples: 3202
Matching samples: 272


In [64]:
ascairn_samples = samplesheet_1kgp[samplesheet_1kgp["SAMPLE_NAME"].isin(matching_samples)]

In [69]:
ascairn_samples[["SAMPLE_NAME", "ENA_FILE_PATH"]].to_csv("ascairn_samples.csv", index=False)